In [1]:
import pandas as pd
import re
import random
from datasets import Dataset
from transformers import AutoTokenizer

# ==========================================
# 1. LOAD DATASET ASLI (Satu-satunya CSV yang dibaca)
# ==========================================
file_name = "../../data/dataset_report_generator.csv" 
try:
    df_asli = pd.read_csv(file_name)
    print(f"✅ Dataset asli dimuat! Total baris: {df_asli.shape[0]}")
except FileNotFoundError:
    print(f"❌ File {file_name} tidak ditemukan.")

# ==========================================
# 2. IN-MEMORY AUGMENTATION (Tanpa Save CSV)
# ==========================================
def augment_permutation(text):
    if pd.isna(text): return text
    pattern = r'(\[[A-Z_]+\])\s*([^\[]+)'
    matches = re.findall(pattern, str(text))
    if not matches: return text
    random.shuffle(matches) # Acak urutan Tag
    return " ".join([f"{tag} {val.strip()}" for tag, val in matches])

print("\n🔄 Melakukan Augmentasi (Feature Permutation) di dalam memori...")
df_augmented = df_asli.copy()
df_augmented['linearized_data_input'] = df_augmented['linearized_data_input'].apply(augment_permutation)

# Gabungkan data asli dengan data hasil acakan (TYPO DIPERBAIKI DI SINI)
df = pd.concat([df_asli, df_augmented], ignore_index=True).drop_duplicates().reset_index(drop=True)
print(f"📈 Jumlah baris data SETELAH Augmentasi: {df.shape[0]} baris")

# ==========================================
# 3. PREPROCESSING & PENAMBAHAN PREFIX T5
# ==========================================
def clean_text_nlg(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\.,!?\-Rp\[\]\|]', '', text)
    return text.strip()

print("\n🧹 Memulai pembersihan teks dan penambahan Prefix...")
# Tambahkan Prefix "Buat laporan: " langsung di sini
prefix_perintah = "Buat laporan: "
df['clean_input'] = prefix_perintah + df['linearized_data_input'].apply(clean_text_nlg)
df['clean_output'] = df['narasi_laporan_output'].apply(clean_text_nlg)

# --- PENAMBAHAN PRINT OUTPUT DI SINI ---
print(f"✨ Preprocessing selesai! Total baris data siap Tokenize: {df.shape[0]} baris")

# Ubah ke HuggingFace Dataset
hf_dataset = Dataset.from_pandas(df[['clean_input', 'clean_output']])

# ==========================================
# 4. TOKENIZATION
# ==========================================
model_checkpoint = "google/mt5-small"
print(f"\n🤖 Memuat tokenizer dari {model_checkpoint}...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False, legacy=False)

def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["clean_input"], 
        max_length=128,      
        truncation=True,     
        padding="max_length" 
    )
    labels = tokenizer(
        text_target=examples["clean_output"], 
        max_length=128, 
        truncation=True, 
        padding="max_length"
    )
    label_ids = []
    for label_seq in labels["input_ids"]:
        label_ids.append([token if token != tokenizer.pad_token_id else -100 for token in label_seq])
        
    model_inputs["labels"] = label_ids
    return model_inputs

print("🔢 Memulai Tokenisasi...")
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)
print("\n✅ SEMUA PERSIAPAN SELESAI! Datamu siap masuk ke cell Training (Epoch 20)!")

d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Dataset asli dimuat! Total baris: 1407

🔄 Melakukan Augmentasi (Feature Permutation) di dalam memori...
📈 Jumlah baris data SETELAH Augmentasi: 2747 baris

🧹 Memulai pembersihan teks dan penambahan Prefix...
✨ Preprocessing selesai! Total baris data siap Tokenize: 2747 baris

🤖 Memuat tokenizer dari google/mt5-small...


🔢 Memulai Tokenisasi...


Map: 100%|██████████| 2747/2747 [00:01<00:00, 2250.43 examples/s]


✅ SEMUA PERSIAPAN SELESAI! Datamu siap masuk ke cell Training (Epoch 20)!


In [2]:
import os
import torch  # <-- WAJIB IMPORT TORCH UNTUK CEK GPU
import mlflow
import dagshub
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from peft import get_peft_model, LoraConfig, TaskType

# ==========================================
# 1. SETUP DAGSHUB & MLFLOW
# ==========================================
print("🔗 Menghubungkan ke DagsHub MLflow...")
dagshub.init(repo_owner='NazeeraAlthea', repo_name='exigen-smart-maintenance', mlflow=True)
mlflow.set_experiment("Report_Generator_mT5_LoRA")

# ==========================================
# 2. SPLIT DATASET (Train & Validation)
# ==========================================
print("🗂️ Membagi data menjadi Train (85%) dan Validation (15%)...")
split_dataset = tokenized_datasets.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

# ==========================================
# 3. LOAD MODEL & DETEKSI CUDA (GPU) ✨
# ==========================================
# Mengecek dan memaksa penggunaan GPU jika tersedia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n" + "="*50)
print(f"🖥️ HARDWARE AKTIF : {str(device).upper()}")
if device.type == 'cuda':
    print(f"🔥 NAMA GPU      : {torch.cuda.get_device_name(0)}")
    print(f"💾 MEMORI GPU    : {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")
else:
    print("⚠️ PERINGATAN: Sistem masih membaca CPU! Training akan sangat lambat.")
print("="*50 + "\n")

model_checkpoint = "google/mt5-small"
print(f"🤖 Memuat Model asli {model_checkpoint}...")
# Memuat model dan langsung melemparnya ke memori GPU (.to(device))
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_checkpoint, 
    use_safetensors=True  # KUNCI PENYEMBUHANNYA DI SINI
).to(device)

print("⚡ Memasang arsitektur LoRA (Low-Rank Adaptation)...")
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,               
    lora_alpha=32,
    lora_dropout=0.1
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters() 

# ==========================================
# 4. TRAINING ARGUMENTS & COLLATOR
# ==========================================
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="../../models/report_cost_generator/mt5-report-generator-logs",
    eval_strategy="epoch",           
    save_strategy="epoch",           
    learning_rate=2e-4,              
    per_device_train_batch_size=4,   
    per_device_eval_batch_size=4,
    num_train_epochs=5,             
    weight_decay=0.01,
    logging_steps=5,                 
    report_to="mlflow",              
    load_best_model_at_end=True,     
)

# Memastikan bahwa Trainer juga sadar dia harus pakai CUDA
print(f"⚙️ Konfirmasi Trainer menggunakan: {training_args.device}")

# ==========================================
# 5. INITIALIZE TRAINER & TRAIN
# ==========================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("\n🚀 MEMULAI PROSES FINE-TUNING DI GPU...")
with mlflow.start_run() as run:
    trainer.train()
    
    # 6. SIMPAN MODEL TERBAIK
    save_path = "../../models/report_cost_generator/best_report_generator_lora"
    
    if not os.path.exists("../../models/report_cost_generator"):
        os.makedirs("../../models/report_cost_generator")
        
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    
    print(f"\n✅ Selesai! Model dan Tokenizer terbaik berhasil disimpan di: {save_path}")
    print(f"📊 Silakan buka Dashboard DagsHub kamu untuk melihat grafik evaluasinya.")

🔗 Menghubungkan ke DagsHub MLflow...


Accessing as NazeeraAlthea

Initialized MLflow to track repo "NazeeraAlthea/exigen-smart-maintenance"

Repository NazeeraAlthea/exigen-smart-maintenance initialized!

🗂️ Membagi data menjadi Train (85%) dan Validation (15%)...

🖥️ HARDWARE AKTIF : CUDA
🔥 NAMA GPU      : NVIDIA GeForce RTX 3060 Laptop GPU
💾 MEMORI GPU    : 6.0 GB

🤖 Memuat Model asli google/mt5-small...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4406.85it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


⚡ Memasang arsitektur LoRA (Low-Rank Adaptation)...
trainable params: 344,064 || all params: 300,520,832 || trainable%: 0.1145
⚙️ Konfirmasi Trainer menggunakan: cuda:0

🚀 MEMULAI PROSES FINE-TUNING DI GPU...


Epoch,Training Loss,Validation Loss
1,3.872396,2.622451
2,3.133523,2.225920
3,3.056362,2.045816
4,2.850197,1.993280
5,2.946693,1.962918



✅ Selesai! Model dan Tokenizer terbaik berhasil disimpan di: ../../models/report_cost_generator/best_report_generator_lora
📊 Silakan buka Dashboard DagsHub kamu untuk melihat grafik evaluasinya.
🏃 View run wise-squirrel-744 at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/2/runs/243f8c71ec81455394056d6bd1b43fe0
🧪 View experiment at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/2


In [2]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

save_path = "../../models/report_cost_generator/best_report_generator_lora"
base_model_name = "google/mt5-small"

print("🤖 Memuat Model AI yang sudah lebih pintar...\n")
tokenizer = AutoTokenizer.from_pretrained(save_path)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name, use_safetensors=True)
model = PeftModel.from_pretrained(base_model, save_path)

def buat_laporan_keuangan(data_prediksi_tabel):
    teks_bersih = re.sub(r'\s+', ' ', str(data_prediksi_tabel))
    teks_bersih = re.sub(r'[^\w\s\.,!?\-Rp\[\]\|]', '', teks_bersih).strip()
    
    # TWEAK 1: Kita perjelas perintahnya agar AI ingat untuk memasukkan biaya
    teks_input_final = "Buat laporan lengkap beserta biaya: " + teks_bersih
    
    inputs = tokenizer(teks_input_final, return_tensors="pt", max_length=128, truncation=True)
    
    # Karena model kita sekarang di GPU, pindahkan juga inputnya ke GPU
    input_ids = inputs["input_ids"].to(model.device)
    
    # TWEAK 2: Parameter Generation yang lebih memaksa
    outputs = model.generate(
        input_ids=input_ids,
        # 1. Batasi panjangnya, ringkasan tidak butuh banyak kata
        max_new_tokens=250,     
        min_new_tokens=100,     
        
        # 2. MATIKAN Beam Search, AKTIFKAN Nucleus Sampling
        do_sample=True,        
        top_p=0.90,             # Mengambil 90% kata paling masuk akal (mencegah kata aneh)
        top_k=50,               # Membatasi pilihan hanya pada 50 kata terbaik
        temperature=0.7,        # Cukup kreatif, tapi tidak halusinasi
        
        # 3. Hukuman Mati untuk Pengulangan
        repetition_penalty=2.5, # Penalti dinaikkan drastis agar dia takut mengulang kata
        no_repeat_ngram_size=2, # Tidak boleh ada 2 kata berurutan yang sama persis diulang
        
        early_stopping=True
    )
    
    laporan = tokenizer.decode(outputs[0], skip_special_tokens=True)
    laporan = re.sub(r'<extra_id_\d+>', '', laporan).strip()
    return laporan

# --- JALANKAN TESTING ---
data_testing_1 = "[MESIN] genset [STATUS] Prediksi Rusak 7 Hari [SEVERITY] Kritis [ESTIMASI_BIAYA] Rp 25.000.000"
print(buat_laporan_keuangan(data_testing_1))

🤖 Memuat Model AI yang sudah lebih pintar...



Loading weights: 100%|██████████| 190/190 [00:00<00:00, 9895.05it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


tampak Kritis dalam 7 hari, dengan sebesar Rp 25.000.000. Estimasi biaya yang diperkirakan adalah Rp 10.000.000 untuk perbaikan dana terjadi diproyeksikan anggaran ini kemungkinan penanganan inaktif akan disiapkan pada genset sebagai keparahan tinggi. Diprediksi kerusakan risiko tidak berpotensi peningsi perlu menyiapan anyaman siaran itu bisa menyelenggara selama 24hari daripada Rp 31.000.000. Angkar persiaga departemen politik mendatang segera berjumlah tingkat kepehas Kritisan.
